# U-Values and Thermal Calculations

This notebook calculates thermal conductivity (lambda) values, R-values, and U-values for building materials, with a focus on External Wall Insulation (EWI) payback analysis.

## Setup: Import Libraries

In [1]:
import pint

ureg = pint.UnitRegistry()
ureg.define('gbp = []')  # Define GBP as a dimensionless currency unit

## Material Properties: Thermal Conductivity (Lambda)

Units: W/mK (watts per metre-kelvin)

In [2]:
LAMBDA = {
    # Insulation
    "xps": 0.04,  # Polyisocyanurate
    # Masonry
    "brick": 0.9,  # Medium density clay brick (~1800 kg/m³)
    "clay_brick_lightweight": 0.5,  # Lightweight clay brick (< 1700 kg/m³)
    # Board materials
    "plasterboard": 0.25,  # Standard plasterboard (gypsum)
}

## Wall Construction Definition

Thicknesses in millimetres

In [3]:
WALL = {
    "plasterboard": 13.0,  # Internal plaster finish
    "brick": 220.0,  # Main structural brick layer
    "xps": 50.0,  # Internal insulation board
}

## Helper Functions

In [4]:
def R_value(element):
    """Calculate R-value (thermal resistance) for a wall element."""
    return (WALL[element] / 1000) / LAMBDA[element]


def lambda_val(material):
    """Get thermal conductivity with units."""
    return LAMBDA[material] * ureg.watts / (ureg.meter * ureg.kelvin)

## Calculate Total R-Value and U-Value for Wall

In [5]:
tot_R = 0.0

print("Wall Construction R-Values:")
print("=" * 40)
for i in WALL:
    R = R_value(i)
    tot_R += R
    print(f"{i:15s}: R = {R:.5f} m²K/W")

print("=" * 40)
print(f"{'Total R-value':15s}: {tot_R:.5f} m²K/W")
print(f"{'U-value':15s}: {1.0 / tot_R:.3f} W/m²K")

Wall Construction R-Values:
plasterboard   : R = 0.05200 m²K/W
brick          : R = 0.24444 m²K/W
xps            : R = 1.25000 m²K/W
Total R-value  : 1.54644 m²K/W
U-value        : 0.647 W/m²K


## External Wall Insulation (EWI) Analysis

In [6]:
# EWI specifications
EWI_thickness = 60 / 1000 * ureg.meter  # 60mm insulation as per Jostec SAP
brick_thickness = (0.220 + 0.025) * ureg.meter  # Brick + ~1 inch solid plaster

print("EWI Payback Analysis")
print("=" * 40)

EWI Payback Analysis


### Calculate R-values for EWI Components

In [7]:
# R-value for existing brick wall
r_brick = brick_thickness / lambda_val("brick")
print(f"R-value for brick wall: {r_brick:.4f}")
print(f"U-value for brick wall: {1.0 / r_brick:.4f} W/m²K")
print()

# R-value for EWI insulation
r_xps = EWI_thickness / lambda_val("xps")
print(f"R-value for EWI (XPS): {r_xps:.4f}")
print(f"U-value for EWI alone: {1.0 / r_xps:.4f} W/m²K")
print()

# Combined R-value with EWI
r_with_ewi = r_brick + r_xps
print(f"Combined R-value (brick + EWI): {r_with_ewi:.4f}")
print(f"Combined U-value (brick + EWI): {1.0 / r_with_ewi:.4f} W/m²K")

R-value for brick wall: 0.2722 kelvin * meter ** 2 / watt
U-value for brick wall: 3.6735 watt / kelvin / meter ** 2 W/m²K

R-value for EWI (XPS): 1.5000 kelvin * meter ** 2 / watt
U-value for EWI alone: 0.6667 watt / kelvin / meter ** 2 W/m²K

Combined R-value (brick + EWI): 1.7722 kelvin * meter ** 2 / watt
Combined U-value (brick + EWI): 0.5643 watt / kelvin / meter ** 2 W/m²K


### Energy and Cost Calculations

In [8]:
# Constants
degree_days = 2250 * ureg.kelvin  # Assumed for Knebworth (base 15°C)
gas_price_bill = 0.03  # £/kWh (2021 pre-crisis price)

# Convert gas price to £/J
gas_price = (
    gas_price_bill / 1000 / (60 * 60) / (ureg.watt * ureg.second) * ureg.gbp
)

day = 24 * 60 * 60 * ureg.seconds

# Calculate U-value improvement
u_delta = 1.0 / r_brick - 1.0 / r_with_ewi

print(f"Change in U-value from EWI: {u_delta:.4f} W/m²K")
print(f"Energy transfer per m² per degree per day: {u_delta:.6f}")
print()

# Calculate savings
daily_saving_psm = day * gas_price * u_delta
print(f"Daily saving per m² per degree day: {daily_saving_psm:.6f}")

year_cost_psm = degree_days * day * gas_price * u_delta
print(f"Annual cost saving per m²: {year_cost_psm:.2f}")

Change in U-value from EWI: 3.1092 watt / kelvin / meter ** 2 W/m²K
Energy transfer per m² per degree per day: 3.109206 watt / kelvin / meter ** 2

Daily saving per m² per degree day: 0.002239 gbp / kelvin / meter ** 2
Annual cost saving per m²: 5.04 gbp / meter ** 2


### Payback Calculation

In [9]:
# Installation cost
# Based on non-binding verbal quote from TAW: £30K for front and side elevations (~150m²)
installation_cost = 150 * ureg.gbp / (ureg.meter**2)  # £/m²

print(f"Installation cost: {installation_cost:.0f}")
print(f"Annual saving per m²: {year_cost_psm:.2f}")
print()

simple_payback = installation_cost / year_cost_psm
print(f"Simple payback period: {simple_payback:.1f} years")


Installation cost: 150 gbp / meter ** 2
Annual saving per m²: 5.04 gbp / meter ** 2

Simple payback period: 29.8 dimensionless years


## Summary

- **Original wall U-value**: ~{:.2f} W/m²K
- **Wall with EWI U-value**: ~{:.2f} W/m²K
- **Estimated payback**: ~{:.0f} years (at 2021 gas prices)

*Note: Payback period will be shorter with current higher gas prices.*